# LangGraph Validation & Conversation Memory

This notebook validates the reusable LangGraph application modules.

It checks:

- Graph compilation
- Node and edge structure
- Intent/safety routing
- Fraud branch when transaction features are supplied
- RAG branch for support queries without transaction data
- SQLite conversation memory

**Gemini-backed cells consume API quota and should be run selectively.**


## 1. Setup


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System


## 2. Import the Production Graph


In [2]:
from src.graph.banking_graph import create_banking_graph

graph = create_banking_graph()

print(type(graph))
print("Graph supports invoke:", hasattr(graph, "invoke"))
print("Graph nodes:")

print(
    list(
        graph.get_graph().nodes.keys()
    )
)


e:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4418.87it/s]


<class 'langgraph.graph.state.CompiledStateGraph'>
Graph supports invoke: True
Graph nodes:
['__start__', 'preprocess', 'intent', 'sentiment', 'priority', 'fraud', 'rag', 'action', 'llm', '__end__']


## 3. Run a Fraud Query with Transaction Data


In [3]:
result = graph.invoke(
    {
        "query": (
            "Someone used my credit card "
            "without my knowledge for ₹5000"
        ),
        "transaction": {
            "amount_inr": 5000,
            "hour_of_day": 23,
            "day_of_week": "Tuesday",
        },
    }
)

print("Intent:", result.get("intent"))
print("ML Intent:", result.get("model_intent"))
print("Fraud Override:", result.get("fraud_override"))
print("Intent Source:", result.get("intent_source"))
print("Sentiment:", result.get("sentiment"))
print("Urgency:", result.get("urgency"))
print("Fraud Probability:", result.get("fraud_probability"))
print("Risk Level:", result.get("risk_level"))
print("Escalate:", result.get("escalate"))
print("Graph Path:", " → ".join(result.get("graph_path", [])))


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Intent: Fraud/Unauthorized
ML Intent: Fraud/Unauthorized
Fraud Override: True
Intent Source: fraud_safety_rule
Sentiment: Urgent
Urgency: High
Fraud Probability: 0.1067
Risk Level: Low
Escalate: False
Graph Path: preprocess → intent → sentiment → fraud → rag → llm


## 4. Run a Fraud-Support Query Without Transaction Data


In [4]:
result = graph.invoke(
    {
        "query": (
            "I received an OTP but never requested it"
        )
    }
)

print("Intent:", result.get("intent"))
print("Fraud Probability:", result.get("fraud_probability"))
print("Risk Level:", result.get("risk_level"))
print("Routing:", result.get("routing_reason"))
print("Graph Path:", " → ".join(result.get("graph_path", [])))


Intent: Fraud/Unauthorized
Fraud Probability: None
Risk Level: None
Routing: Fraud-related request detected. Transaction risk scoring was not performed because the required transaction features (amount, transaction hour, and day of week) were not fully available.
Graph Path: preprocess → intent → sentiment → rag → llm


## 5. Run a KYC Query


In [5]:
result = graph.invoke(
    {
        "query": (
            "My Aadhaar verification is failing. "
            "What documents do I need?"
        )
    }
)

print("Intent:", result.get("intent"))
print("Sentiment:", result.get("sentiment"))
print("Urgency:", result.get("urgency"))
print("Policy sources:")

for item in result.get("policy_results", []):
    print(
        item["source"],
        item["similarity_score"],
    )


Intent: KYC
Sentiment: Confused
Urgency: Medium
Policy sources:
kyc_policy.txt 0.745
kyc_policy.txt 0.6603
kyc_policy.txt 0.5805


## 6. SQLite Conversation Memory


In [6]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "models"
    / "banking_checkpoints.sqlite"
)

connection = sqlite3.connect(
    str(CHECKPOINT_PATH),
    check_same_thread=False,
)

checkpointer = SqliteSaver(
    connection
)

memory_graph = create_banking_graph(
    checkpointer=checkpointer
)

thread_config = {
    "configurable": {
        "thread_id": "notebook_memory_validation"
    }
}


In [7]:
first_result = memory_graph.invoke(
    {
        "query": (
            "I see an unauthorized transaction "
            "of ₹15000 on my account."
        ),
        "transaction": {
            "amount_inr": 15000,
            "hour_of_day": 14,
            "day_of_week": "Monday",
        },
    },
    thread_config,
)

second_result = memory_graph.invoke(
    {
        "query": "What should I do next?"
    },
    thread_config,
)

print("First intent:", first_result.get("intent"))
print("Second intent:", second_result.get("intent"))

saved_state = memory_graph.get_state(
    thread_config
)

print(
    "Persisted messages:",
    len(
        saved_state.values.get(
            "messages",
            []
        )
    )
)

for message in saved_state.values.get(
    "messages",
    []
):
    print(
        f"{message.type}: {message.content}"
    )


First intent: Fraud/Unauthorized
Second intent: Fraud/Unauthorized
Persisted messages: 4
human: I see an unauthorized transaction of ₹15000 on my account.
ai: I am very sorry to hear about the unauthorized transaction on your account. Given the urgency of this situation, please take the following steps immediately to protect your account:

*   **Block your card:** Please block all your payment instruments immediately. You can do this through our mobile banking application under the "Report Fraud" section, or by calling our 24x7 toll-free helpline at 1800-XXX-XXXX.
*   **Report the transaction:** It is essential to report this unauthorized activity as soon as possible. In addition to the mobile app or helpline, you may also visit your nearest branch or submit a written complaint to the nodal officer.
*   **Reset Credentials:** Please reset your net banking and mobile banking credentials as a security precaution.

Please be aware that under our fraud policy, reporting unauthorized transa

## 7. Final Validation Checklist


In [8]:
checks = {
    "compiled_graph": hasattr(graph, "invoke"),
    "graph_inspection": hasattr(graph, "get_graph"),
    "fraud_model_result_present": (
        result is not None
    ),
    "sqlite_database_exists": CHECKPOINT_PATH.exists(),
}

for name, passed in checks.items():
    print(
        f"{'PASS' if passed else 'CHECK'}: {name}"
    )


PASS: compiled_graph
PASS: graph_inspection
PASS: fraud_model_result_present
PASS: sqlite_database_exists


## Final Result

The notebook validates the reusable production graph rather than duplicating node implementation.

Production modules remain under:

- `src/graph/state.py`
- `src/graph/banking_graph.py`
- `src/ml/intent.py`
- `src/ml/fraud.py`
- `src/rag/retriever.py`
- `src/llm/gemini.py`
